### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="colon_tumor",
    dataset_year="1999",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="OpenML",
    original_dataset_source_download_link="https://www.openml.org/d/45087",
    download_description="""
We download the data to a predefined folder.

mkdir -p local-data-warehouse/colon_tumor && wget -P local-data-warehouse/colon_tumor https://api.openml.org/data/download/22112148/dataset 
""",
    # References
    academic_reference_bibtex="""@article{Alon1999BroadPO,
  title={Broad patterns of gene expression revealed by clustering analysis of tumor and normal colon tissues probed by oligonucleotide arrays},
  author={Uri Alon and Naama Barkai and Daniel A. Notterman and Kurt Gish and Suzanne Ybarra and Daniel Mack and Arnold J. Levine},
  journal={Proceedings of the National Academy of Sciences},
  year={1999},
  volume={96},
  number={12},
  pages={6745-6750},
  doi={10.1073/pnas.96.12.6745},
  url={https://doi.org/10.1073/pnas.96.12.6745}
}
""",
    academic_reference_bibtex_key="Alon1999BroadPO",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
        - We use the OpenML version of the data as original raw data in not available. The features have been pre-filtered in an unsupervised way from 6,500 to 2,000. They have been already z-normalized.
        - We remove 9 duplicated columns: "att_261", "att_262", "att_263", "att_51", "att_52", "att_53", "att_40", "att_41", "att_42".
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="HasTumor",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="HasTumor",
)

## Preprocessing

In [2]:
import pandas as pd
import arff

with open(dataset_mold.path/"dataset") as f:
    data = arff.load(f)
df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

target_feature = "HasTumor"
df.rename(columns={"class": target_feature}, inplace=True)
df[target_feature] = df[target_feature].map({"-1": "Yes", "1": "No"})
df.drop(columns=["att_261", "att_262", "att_263", "att_51", "att_52", "att_53", "att_40", "att_41", "att_42"], inplace=True)
df["HasTumor"] = df["HasTumor"].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 62
Columns: 1992
Use sampling: False (sample size: 62)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['att_1005', 'att_1333', 'att_1346', 'att_1345', 'att_1344', 'att_1343', 'att_1342', 'att_1341', 'att_1340', 'att_1339']
Rows remaining as candidates after top-10 filter: 0 (of 62)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

att_1     att_2     att_3     att_4     att_5     att_6     att_7  \
0  2.080750  1.099070  0.927763  1.029080 -0.130763  1.265460 -0.436286   
1  1.109460  0.786453  0.445560 -0.146323 -0.996316  0.555759  0.290734   
2 -0.676530  1.693100  1.559250  1.559980 -0.982179 -1.358510 -1.313990   
3  0.534396  1.677540  1.489030  0.778605 -0.183776 -1.116850 -1.487560   
4 -1.018900  0.511080  0.755641  1.013820  0.529899  0.160440 -0.087055   

      att_8     att_9    att_10    att_11    att_12    att_13    att_14  \
0  0.728881  2.107980  1.359870  0.265471 -0.324436 -1.200930 -1.531920   
1 -0.145259  1.132660  0.559093 -1.469130 -0.813699  0.594290  0.404811   
2 -0.455067  0.295214  0.290694  0.415632  0.136703 -0.688848 -1.700350   
3 -0.579511  0.292683  1.345480 -0.687898 -0.749802 -1.168640 -0.559310   
4  1.295290  0.458736  0.714082  0.727290  1.558500  1.340430  0.486905   

     att_15    att_16    att_17    att_18    att_19    att_20    att_21  \
0  0.298026  0.088341  0.644631 -0.650944 -0.891003 -0.185717  1.395990   
1 -1.197510  0.947450 -1.231180 -0.364203 -0.248986 -0.376637  0.589491   
2 -1.864170  0.504606 -2.122360  0.407008 -0.780883 -1.336130 -0.766691   
3 -0.818024  0.239036 -1.534590 -0.126724 -1.172270 -1.586910  0.429054   
4  0.876190  0.502442 -0.657175  1.140000  0.104323  1.278850 -1.205270   

     att_22    att_23    att_24    att_25    att_26    att_27    att_28  \
0 -0.832146  2.115500  1.347400  0.319671  0.206106 -0.377173 -1.607610   
1 -0.196278  0.614148  0.799198  0.479413 -0.970803 -0.875199 -1.878890   
2 -3.344640 -0.195161  0.210595 -0.246315 -1.299920  0.242096 -2.117990   
3 -0.719144  0.607006  1.399280 -0.676216 -0.657361 -0.523732 -1.562170   
4  1.254230 -0.024230  0.435259  1.132150  0.716794  1.609960  0.622332   

     att_29    att_30    att_31    att_32    att_33    att_34    att_35  \
0  0.856661 -0.208192 -0.227354  0.455312  0.700828  1.460820 -0.127914   
1  0.511563 -1.740870 -1.455500  1.448560  0.623051  1.141090  0.385889   
2 -1.217960 -1.473650 -1.879480 -0.669519  0.376670  0.948078  0.327035   
3 -1.212070 -1.624320 -0.608500  2.018430  1.611020  1.995410 -0.229551   
4  0.268844 -0.128929  1.248860 -0.133293 -0.851664 -0.004749  1.489870   

     att_36    att_37    att_38    att_39    att_43    att_44    att_45  \
0  1.686890 -1.400370  1.306270  0.640420  0.584761  0.777720  0.946010   
1  0.569032 -1.935770  0.964068  0.571525 -1.223540  1.455530  0.472138   
2  0.149636 -1.117270  1.503350  0.121324 -1.766400  0.497568 -0.562133   
3  0.390630 -0.404622  2.140690  0.987880 -1.190700  1.830350 -0.899369   
4 -1.814500  0.156711 -0.677027 -0.865020  1.542100 -0.421385  0.984692   

     att_46    att_47    att_48    att_49    att_50    att_54    att_55  \
0  1.448660 -0.016243 -0.516636  0.150824  1.572940 -0.631182  0.252795   
1  0.715017 -0.863870  0.685510  1.777260 -1.118940 -0.678083  0.975620   
2  0.369011 -1.294920 -1.295150  1.048160 -0.329538 -1.364490 -0.357240   
3  1.020460 -0.377963 -1.324830  1.396010  0.286153 -1.833830  0.997750   
4 -1.467550  0.540824  0.394370  0.854811  0.041680  1.660830 -0.841720   

     att_56    att_57    att_58    att_59    att_60    att_61    att_62  \
0 -0.355982 -1.897550 -0.433645  1.994650 -0.018567  0.061276 -0.604689   
1  0.462957 -0.793426  0.310381  1.123670 -0.420843 -1.092250 -0.912311   
2 -2.553610 -1.067760 -1.462910 -0.635544 -1.332990 -1.721450 -2.040360   
3 -0.778719 -0.201567 -1.984640  0.010463 -0.765827 -1.490820 -1.091350   
4  0.710755  0.374812 -0.210729 -1.404720  0.504688  1.072960  2.141230   

     att_63    att_64    att_65    att_66    att_67    att_68    att_69  \
0  0.905910 -0.782365  0.964356  0.016980  0.116946 -0.733182 -0.468315   
1  0.258039  0.364651  0.574631  0.639241  0.684171  0.526483  0.074961   
2  0.132054 -0.824488 -0.954626 -0.913681 -1.372790 -1.798860 -1.247290   
3  0.532486 -1.382160  0.146811 -1.262440 -0.818342 -1.193010 -1.393770   
4 -2.349050  0.390752 -

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,HasTumor,category,0.0,0.0,2.0,"Yes, No"
1,att_1,float64,0.0,0.0,62.0,"2.0808, -2.3456, 0.4512, 0.1736, -0.5528, -1.6337, -1.1546, -1.7652, 1.4834, -0.953"
2,att_2,float64,0.0,0.0,62.0,"1.0991, -1.2664, 1.5331, -0.087, -0.395, 1.225, -1.1395, 0.4341, 0.5672, 1.1079"
3,att_3,float64,0.0,0.0,62.0,"0.9278, -1.2953, 1.4932, 0.0493, -0.0674, 1.0365, -0.8528, 0.3975, 0.5733, 1.3285"
4,att_4,float64,0.0,0.0,62.0,"1.0291, -1.8666, 0.1087, 0.1546, -1.4685, 1.1086, -1.8056, -2.1802, -0.2029, -1.6446"
5,att_5,float64,0.0,0.0,62.0,"-0.1308, -0.6652, 0.1187, -0.8546, 0.9867, -2.1212, 1.2116, -0.4822, 1.1082, -0.0742"
6,att_6,float64,0.0,0.0,62.0,"1.2655, -0.3256, 0.3596, 0.3957, -0.951, 0.0875, -0.0432, 0.5799, 1.6272, -0.7767"
7,att_7,float64,0.0,0.0,62.0,"-0.4363, -0.1646, -0.0807, -0.5017, -0.5648, 0.2854, 1.0209, 0.0072, 1.2024, -1.4058"
8,att_8,float64,0.0,0.0,62.0,"0.7289, -0.1915, 1.088, -0.6068, 0.9063, -0.4536, 0.4752, -0.3964, -0.094, -0.3387"
9,att_9,float64,0.0,0.0,62.0,"2.108, -1.8555, 0.0418, 0.4277, -0.3363, 0.2922, -0.7338, -2.1862, 0.7798, -0.5836"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
att_1,62.0,1.774194e-07,1.000000,-2.34557,2.08075
att_2,62.0,9.677419e-08,1.000000,-2.62049,1.69310
att_3,62.0,-1.451613e-07,1.000000,-2.76391,1.55925
att_4,62.0,8.064516e-08,1.000000,-2.18024,2.52569
att_5,62.0,4.838710e-08,1.000000,-2.12120,2.89500
att_6,62.0,-3.225806e-08,1.000000,-2.98831,1.62717
att_7,62.0,6.451613e-08,1.000000,-2.34981,1.74813
att_8,62.0,9.677419e-08,1.000000,-2.44796,2.68992
att_9,62.0,-4.838710e-08,1.000000,-2.92951,2.10798
att_10,62.0,9.677419e-08,1.000000,-2.67227,2.25484


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column   rank                    
HasTumor 1      Yes     40  64.52
         2       No     22  35.48

In [8]:
# Target Distribution
target_df

,count,pct
HasTumor,,
Yes,40,64.52
No,22,35.48


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to colon_tumor/019d9cd6-7112-7d0e-a09d-f0e137b371cf
019d9cd6-7112-7d0e-a09d-f0e137b371cf
8115e47a3b674163b4219806397cba1b708cb26217ba509d4a659bb14a5ad0cc
